In [36]:
import torch

In [37]:
torch.set_default_dtype(torch.float64)

In [38]:
m = 7
l = 9
vocab_size = 9
target = [0, 1, 2, 3, 4, 5, 8]
assert len(target) == m
target = torch.tensor(target)
p = 3
p_idx = -1
mstar = m + p
lstar = l + p
target = torch.cat([target, torch.tensor([p_idx] * p)])

In [39]:
target

tensor([ 0,  1,  2,  3,  4,  5,  8, -1, -1, -1])

In [40]:
transition_matrix = torch.zeros((lstar, lstar))

In [41]:
transition_matrix.shape

torch.Size([12, 12])

In [42]:
# the following are the coordinates and values, using 1-indexing
transitions = [
    (
        (1, 2), #row, col
        0.3 #prob
    ),
    (
        (1, 3), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        1.0 #prob
    ),
    (
        (3, 4), #row, col
        1.0 #prob
    ),
    (
        (4, 5), #row, col
        1.0 #prob
    ),
    (
        (5, 6), #row, col
        0.5 #prob
    ),
    (
        (5, 7), #row, col
        0.5 #prob
    ),
    (
        (6, 9), #row, col nice
        1.0 #prob
    ),
    (
        (7, 8), #row, col
        1.0 #prob
    ),
    (
        (8, 9), #row, col
        1.0 #prob
    ),

    ## these transitions are dummy transitions for the padding
    ## for any (x,y), we must have 9 <= x and 10 <= y
    ## and y >= x + 1
    (
        (9, 10), #row, col
        0.5 #prob
    ),
    (
        (9, 11), #row, col
        0.5 #prob
    ),
    (
        (10, 12), #row, col
        1.0 #prob
    ),
    (
        (11, 12), #row, col
        1.0 #prob
    )
]

In [43]:
for (row, col), prob in transitions:
    transition_matrix[row-1, col-1] = prob

In [44]:
transition_matrix

tensor([[0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,

In [45]:
transition_matrix[m]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])

In [46]:
token_probs = torch.zeros((lstar, vocab_size))

In [47]:
# the following are the coordinates and values, using 1-indexing
# format is
# (row, col), value
# for example (1, 2), 0.8
# means state 1 emits token 2 with probability 0.8
emission_probs = [
    (
        (1, 1), #row, col
        0.9 #prob
    ),
    (
        (1, 5), #row, col
        0.1 #prob
    ),
    (
        (2, 2), #row, col
        0.7 #prob
    ),
    (
        (2, 3), #row, col
        0.3 #prob
    ),
    (
        (3, 2), #row, col
        0.2 #prob
    ),
    (
        (3, 3), #row, col
        0.8 #prob
    ),
    (
        (4, 3), #row, col
        0.1 #prob
    ),
    (
        (4, 4), #row, col
        0.9 #prob
    ),
    (
        (5, 4), #row, col
        0.1 #prob
    ),
    (
        (5, 5), #row, col
        0.9 #prob
    ),
    (
        (6, 6), #row, col
        0.6 #prob
    ),
    (
        (6, 7), #row, col
        0.3 #prob
    ),
    (
        (6, 8), #row, col
        0.1 #prob
    ),
    (
        (7, 5), #row, col
        0.1 #prob
    ),
    (
        (7, 6), #row, col
        0.1 #prob
    ),
    (
        (7, 7), #row, col
        0.7 #prob
    ),
    (
        (7, 2), #row, col
        0.1 #prob
    ),
    (
        (8, 6), #row, col
        0.2 #prob
    ),
    (
        (8, 8), #row, col
        0.6 #prob
    ),
    (
        (8, 9), #row, col
        0.2 #prob
    ),
    (
        (9, 8), #row, col
        0.3 #prob
    ),
    (
        (9, 9), #row, col
        0.7 #prob
    ),
    ## padding token emissions
    ## for any (x, y), we must have x >= 10 and 1 <= y <= 9
    (
        (10, 1), #row, col
        0.1 #prob
    ),
    (
        (10, 3), #row, col
        0.9 #prob
    ),
    (
        (11, 4), #row, col
        1.0 #prob
    ),
    (
        (12, 5), #row, col
        1.0 #prob
    )
]

In [48]:
for (row, col), prob in emission_probs:
    token_probs[row-1, col-1] = prob

In [49]:
transition_matrix

tensor([[0.0000, 0.3000, 0.7000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5000, 0.5000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,

In [50]:
# turn transition matrix, token probs, and dp into log space
transition_matrix = torch.log(transition_matrix)
token_probs = torch.log(token_probs)

In [51]:
l = transition_matrix.shape[0]

In [52]:
tokens = torch.argmax(token_probs, dim=1)

In [53]:
edges = torch.argmax(transition_matrix, dim=1)

In [54]:
out_tokens = [tokens[0].item()]

In [55]:
tokens

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 2, 3, 4])

In [56]:
edges

tensor([ 2,  2,  3,  4,  5,  8,  7,  8,  9, 11, 11,  0])

In [57]:
mini = 0

In [58]:
i = 0
while i < l:
    i = edges[i].item()
    mini = max(mini, i)
    if i >= l or i < mini:
        break
    out_tokens.append(tokens[i].item())

In [59]:
out_tokens

[0, 2, 3, 4, 5, 8, 2, 4]